Log Probability func implementation


In [22]:
import math as m

# Define HMM states (including artificial start/end) just to make things easier
# We can also do it with three states but it would reuire little extra changes in final answer
states = ['start', 'E', '5', 'I', 'end']
# transition probabilites
trans_probab = {
    'start': {'E': m.log(1.0)},
    'E':     {'E': m.log(0.9), '5': m.log(0.1)},
    '5':     {'I': m.log(1.0)},
    'I':     {'I': m.log(0.9), 'end': m.log(0.1)},
}

# emission probabilites
emiss_probab = {
    'E': {'A': m.log(0.25), 'C': m.log(0.25), 'G': m.log(0.25), 'T': m.log(0.25)},
    '5': {'A': m.log(0.05), 'G': m.log(0.95)},
    'I': {'A': m.log(0.4),  'C': m.log(0.1),  'G': m.log(0.1),  'T': m.log(0.4)},
}

def prob_of_a_given_path(Seq: str, Path: str):

    if len(Seq) != len(Path):
        raise ValueError("Lengths must match")

    ans = 0.0

    # Transition from 'start' → first state + that state's emission
    first = Path[0]
    ans += trans_probab['start'].get(first, float('-inf'))
    ans += emiss_probab[first].get(Seq[0], float('-inf'))

    for idx in range(1, len(Seq)):
        prev_state = Path[idx - 1]
        curr_state = Path[idx]
        symbol     = Seq[idx]

        ans += trans_probab[prev_state].get(curr_state, float('-inf'))
        ans += emiss_probab[curr_state].get(symbol, float('-inf'))

    # Final transition from last state → 'end'
    last = Path[-1]
    ans += trans_probab[last].get('end', float('-inf'))

    return ans

# Example provided
seq  = "CTTCATGTGAAAGCAGACGTAAGTCA"
path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
score = prob_of_a_given_path(seq, path)
print("Ans: {:.2f}".format(score))




Ans: -41.22


Viterbi Implementation

In [21]:
# I have Taken Help for the viterbi implementation part from Geek for Geeks
import math

# Define HMM states (including artificial start/end)
states = ['start', 'E', '5', 'I', 'end']

# Transition log-probabilities
trans_probab = {
    'start': {'E': math.log(1.0)},
    'E':     {'E': math.log(0.9),  '5': math.log(0.1)},
    '5':     {'I': math.log(1.0)},
    'I':     {'I': math.log(0.9),  'end': math.log(0.1)},
}

# Emission log-probabilities
emiss_probab = {
    'E': {'A': math.log(0.25), 'C': math.log(0.25), 'G': math.log(0.25), 'T': math.log(0.25)},
    '5': {'A': math.log(0.05), 'G': math.log(0.95)},
    'I': {'A': math.log(0.4),  'C': math.log(0.1),  'G': math.log(0.1),  'T': math.log(0.4)},
}

# Intitailizing the Viterbi table and backpointer table
def initialize_viterbi(sequence: str):
    length = len(sequence)
    viterbi = [ {st: float('-inf') for st in states} for _ in range(length) ]
    backptr = [ {} for _ in range(length) ]

    first_symbol = sequence[0]
    viterbi[0]['E'] = emiss_probab['E'][first_symbol]

    return viterbi, backptr

# Actual Implementation begins :
def Viterbi(sequence: str, viterbi: list, backptr: list):

    for i in range(1, len(sequence)):
        obs = sequence[i]
        for curr_st in states:
            best_score = float('-inf')
            best_prev = None

            # Evaluate each possible previous state
            for prev_st in states:
                prev_score  = viterbi[i-1].get(prev_st, float('-inf'))
                trans_score = trans_probab.get(prev_st, {}).get(curr_st, float('-inf'))
                emit_score  = emiss_probab.get(curr_st, {}).get(obs, float('-inf'))

                score = prev_score + trans_score + emit_score
                if score > best_score:
                    best_score = score
                    best_prev = prev_st

            viterbi[i][curr_st] = best_score
            backptr[i][curr_st] = best_prev

# Backtracking to find the best path
def backtrack_path(sequence: str, viterbi: list, backptr: list):

    last_idx = len(sequence) - 1
    best_state = max(states, key=lambda st: viterbi[last_idx][st])

    # Walking backwards through the bacptr table
    path = [best_state]
    for pos in range(last_idx, 0, -1):
        prev_state = backptr[pos][path[0]]
        path.insert(0, prev_state)

    return "".join(path)

# Taken an random sequence
sequence = "AGGTAATGATTCGCA"

viterbi_table, backpointer_table = initialize_viterbi(sequence)
Viterbi(sequence, viterbi_table, backpointer_table)
best_path = backtrack_path(sequence, viterbi_table, backpointer_table)

print("Most likely state sequence:", best_path)


Most likely state sequence: EEEEEEEEEEEEEEE
